In [1]:
import numpy as np
import pandas as pd

In [202]:
df = pd.read_csv("data/adjusted_datasets_v33.csv", converters={"Indicator Number": str})
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
len(df)

64214

In [203]:
clusters_df = pd.read_excel('data/innovation_list_HWLclusters_v3.0.xlsx', sheet_name=0)
clusters_df = clusters_df.loc[clusters_df['timeseries']==1]
len(clusters_df)

45

In [207]:
set(df['Innovation Name']).difference(set(clusters_df['innovation_name']))

{'climate protest',
 'drivers licence (not having)',
 'firm ESG reporting',
 'microfinance',
 'mobesity',
 'passive buildings',
 'postage stamps',
 'sustainable fashion',
 'textile recycling',
 'train travel'}

In [210]:
assert len(set(clusters_df['innovation_name']).difference(set(df['Innovation Name']))) == 0

In [206]:
# Correct digital cluster naming
df.loc[df['Innovation Name']=='Cloud computing & SaaS', 'Innovation Name'] = 'cloud computing & SaaS'
df.loc[df['Innovation Name']=='Digital (non-cash) payments', 'Innovation Name'] = 'digital (non-cash) payments'
df.loc[df['Innovation Name']=='Digital skills', 'Innovation Name'] = 'digital skills'
df.loc[df['Innovation Name']=='E-commerce', 'Innovation Name'] = 'e-commerce'
df.loc[df['Innovation Name']=='E-government', 'Innovation Name'] = 'e-government'
df.loc[df['Innovation Name']=='Email', 'Innovation Name'] = 'email'
df.loc[df['Innovation Name']=='Household internet access', 'Innovation Name'] = 'household internet access'
df.loc[df['Innovation Name']=='Mobile broadband', 'Innovation Name'] = 'mobile broadband'
df.loc[df['Innovation Name']=='Mobile/cellphone', 'Innovation Name'] = 'mobile/cellphone'
df.loc[df['Innovation Name']=='Personal computer', 'Innovation Name'] = 'personal computer'
df.loc[df['Innovation Name']=='Social media', 'Innovation Name'] = 'social media'
df.loc[df['Innovation Name']=='World wide web', 'Innovation Name'] = 'world wide web'
df.loc[df['Innovation Name']=='balcony solar ', 'Innovation Name'] = 'balcony solar'
df.loc[df['Innovation Name']=='households electricity access', 'Innovation Name'] = 'household electricity access'

In [209]:
# Drop all innovations that are not in innovation list
df = df.loc[df['Innovation Name'].isin(clusters_df['innovation_name'])]
len(df)

51366

In [211]:
summary_df = pd.read_csv("data/summary_table_v33.csv", converters={"Indicator Number": str})

In [212]:
# Create unique ID
group_vars = ['Innovation Name', 'Spatial Scale', 'Description', 'Metric'] # defines one time series
sep = ' - '
df['name'] = df[group_vars[0]] + sep + df[group_vars[1]]
df['ID'] = df[group_vars[0]]
summary_df['name'] = summary_df[group_vars[0]]
for i in range(1, len(group_vars)):
    df['ID'] += sep + df[group_vars[i]]
    summary_df['name'] += sep + summary_df[group_vars[i]]
len(summary_df['name'].unique())

1786

In [213]:
# Keep only selected and new time series
selection_dict = summary_df.set_index('name')['select_1.1_allregions_FIN']#.fillna(0)
df = df.loc[(df['Indicator Number']!='1.1') |
    ((df['ID'].map(selection_dict) == 1) # take all with 1 in selection column
    | (df['ID'].map(selection_dict).isna()))] # take all that don't appear in selection column
len(df)

49664

In [214]:
len(df['ID'].unique())

1602

In [215]:
len(df.loc[df['Indicator Number']=='1.1', 'ID'].unique())

525

In [216]:
# Find innovations with multiple timeseries
s = df.loc[df['Indicator Number']=='1.1'].groupby('name')['ID'].unique().to_frame(name='ID')
s = s.loc[s['ID'].apply(lambda l: len(l)>1)]
s

,ID
name,
active mobility - Amsterdam,[active mobility - Amsterdam - Bike ownership ...
active mobility - Beijing,[active mobility - Beijing - Bicycle modal sha...
active mobility - The Netherlands,[active mobility - The Netherlands - Passenger...
car ownership (foregoing) - Berlin,[car ownership (foregoing) - Berlin - Berlin C...
car ownership (foregoing) - Heidelberg,[car ownership (foregoing) - Heidelberg - Heid...
eating less meat - China,[eating less meat - China - per capita total m...
eating less meat - India,[eating less meat - India - % red in total mea...
eating less meat - The Netherlands,[eating less meat - The Netherlands - % red in...
net metering - United States,[net metering - United States - Total net mete...


In [217]:
ts_dict = {
    'active mobility - Amsterdam': 'active mobility - Amsterdam - % trips by walking and biking EXCLUDING 1930 - market share',
    'active mobility - Beijing': 'active mobility - Beijing - % trips by walking and biking - market share',
    'active mobility - The Netherlands': 'active mobility - The Netherlands - % trips by walking and biking - market share',
    'car ownership (foregoing) - Berlin': 'car ownership (foregoing) - Berlin - cars per person PARTIAL FROM MAX ONWARDS - market share',
    'car ownership (foregoing) - Heidelberg': 'car ownership (foregoing) - Heidelberg - cars per person - market share',
    'eating less meat - China': 'eating less meat - China - % red in total meat consumption - % kg/yr',
    'eating less meat - India': 'eating less meat - India - % red in total meat consumption - % kg/yr',
    'eating less meat - The Netherlands': 'eating less meat - The Netherlands - % red in total meat consumption - % kg/yr',
    'net metering - United States': 'net metering - United States - Total net metering customers as share of total U.S. households - Total net metering customers / total households',
    'organic food - Austria': 'organic food - Austria - Organic retail sales share [%] - %',
    'organic food - Canada': 'organic food - Canada - Organic retail sales share [%] - %',
    'organic food - Denmark': 'organic food - Denmark - Organic retail sales share [%] - %',
    'organic food - Japan': 'organic food - Japan - Organic retail sales share [%] - %',
    'organic food - Switzerland': 'organic food - Switzerland - Organic retail sales share [%] - %',
    'organic food - The Netherlands': 'organic food - The Netherlands - Organic retail sales share [%] - %',
    'organic food - UK': 'organic food - UK - Organic retail sales share [%] - %',
    'third-party solar leasing - California': 'nan',
    'third-party solar leasing - Connecticut': 'nan',
    'third-party solar leasing - Massachusetts': 'nan',
    'third-party solar leasing - New Jersey': 'nan',
    'third-party solar leasing - US': 'nan',
    'wearables - China': 'wearables - China - % penetration rate total (smartwatch, bands, scales)  - % penetration rate ',
    'wearables - Global': 'wearables - Global - % penetration rate total (smartwatch, bands, scales)  - % penetration rate ',
    'wearables - Japan ': 'wearables - Japan  - % penetration rate total (smartwatch, bands, scales)  - % penetration rate ',
    'wearables - The Netherlands ': 'wearables - The Netherlands  - % penetration rate total (smartwatch, bands, scales)  - % penetration rate '
}

In [218]:
# drop non-selected time series data
for name, ts_id in ts_dict.items():
    if ts_id != 'nan':
        df = df.loc[~((df['Indicator Number']=='1.1') & (df['name']==name) & (df['ID']!=ts_id))]
    else:
        df = df.loc[~((df['Indicator Number']=='1.1') & (df['name']==name))]
len(df)

49020

In [219]:
# Correct wrong indicator coding
df.loc[df['Innovation Name']=='third-party solar leasing', 'Indicator Number'] = '1.1'

In [220]:
len(df['ID'].unique())

1563

In [221]:
len(df.loc[df['Indicator Number']=='1.1', 'ID'].unique())

518

In [222]:
# Save cleansed data
df.drop(['name', 'ID'], axis=1).to_csv("data/adjusted_datasets_v34.csv", index=False)